# VitiGuard MD — дообучение модели диагностики на полевых снимках (ml/fine-02)

Ноутбук дообучает классификатор листьев (EfficientNet-B0) с добавлением индийского полевого набора GVLiD (CC BY 4.0, Махараштра):

- пересобирает `splits.csv` с учётом виноградников GVLiD (без утечки соседних кадров куста);
- дообучает EfficientNet-B0 с аугментациями полевой съёмки (`configs/field.yaml`);
- оценивает модель на тестовой выборке, на полевом тесте HERMOS и на отложенных снимках GVLiD (`gvlid_test`);
- сравнивает метрики с базовой версией и сохраняет `results_field.zip` на Google Диск.

**Перед запуском:**

1. Включите GPU: «Среда выполнения» → «Сменить среду выполнения» → T4 GPU.
2. Убедитесь, что архив `vitiguard_ml_field.zip` лежит в `Мой диск/vitiguard/vitiguard_ml_field.zip`, а базовые результаты — в `Мой диск/vitiguard/results/results.json`.
3. «Среда выполнения» → «Выполнить все».

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/vitiguard"
RESULTS = f"{DRIVE}/results"

Данные распаковываются на локальный диск Colab. При повторном запуске ядро сначала переходит в `/content`, чтобы не удалять рабочий каталог, в котором оно находится.

In [ ]:
# Повторный запуск: нельзя удалять рабочую папку, пока в ней находится ядро
%cd /content
!rm -rf /content/vitiguard && mkdir -p /content/vitiguard
!unzip -q "{DRIVE}/vitiguard_ml_field.zip" -d /content/vitiguard
%cd /content/vitiguard/ml
!pip install -q -e . albumentations timm grad-cam segmentation-models-pytorch opencv-python-headless "onnx<1.23"

import os

# Команды «!python -m ...» запускаются отдельными процессами и сами пакет не видят
os.environ["PYTHONPATH"] = "/content/vitiguard/ml/src"


## 1. Пересборка разбиения с учётом виноградников GVLiD

Все кадры одного виноградника попадают в одну выборку, исключая утечку соседних снимков одного куста между обучением и тестом.

In [ ]:
!python -m vitiguard_ml.data.split --config configs/field.yaml

## 2. Дообучение EfficientNet-B0 и экспорт лучшей модели в ONNX

In [ ]:
!python -m vitiguard_ml.experiments --config configs/field.yaml --out "{RESULTS}/field" --stages classify export --models efficientnet_b0

## 3. Сравнение с базовой версией

In [ ]:
import json
from pathlib import Path

base_file = Path(RESULTS) / "results.json"
field_file = Path(RESULTS) / "field" / "results.json"

base_m = {}
if base_file.exists():
    base_data = json.loads(base_file.read_text(encoding="utf-8"))
    base_m = base_data.get("classification", {}).get("efficientnet_b0", {})
else:
    print(f"Предупреждение: базовый файл {base_file} не найден")

field_data = json.loads(field_file.read_text(encoding="utf-8"))
field_m = field_data.get("classification", {}).get("efficientnet_b0", {})

print(f"{'Метрика':<28}{'Базовая':>12}{'Полевая':>12}{'Разница':>10}")
print("-" * 62)

metrics = [
    ("Test Accuracy", "test", "accuracy"),
    ("Test Macro-F1", "test", "macro_f1"),
    ("HERMOS (поле) Accuracy", "field", "accuracy"),
    ("HERMOS (поле) Macro-F1", "field", "macro_f1"),
    ("GVLiD Test Accuracy", "gvlid_test", "accuracy"),
    ("GVLiD Test Macro-F1", "gvlid_test", "macro_f1"),
]

for title, group, key in metrics:
    b_val = base_m.get(group, {}).get(key) if base_m else None
    f_val = field_m.get(group, {}).get(key) if field_m else None
    b_str = f"{b_val:6.3f}" if b_val is not None else "  —   "
    f_str = f"{f_val:6.3f}" if f_val is not None else "  —   "
    diff_str = f"{f_val - b_val:+6.3f}" if (b_val is not None and f_val is not None) else "  —   "
    print(f"{title:<28}{b_str:>12}{f_str:>12}{diff_str:>10}")

## 4. Сохранение результатов на Google Диск

In [ ]:
!cd "{RESULTS}/field" && rm -f "{DRIVE}/results_field.zip" && zip -qr "{DRIVE}/results_field.zip" results.json figures models
print("Результаты сохранены в", f"{DRIVE}/results_field.zip")